# Homework 2: One Dataframe.ipynb

##Question 1: [IPO] Withdrawn IPOs by Company Type


What is the total withdrawn IPO value (in $ millions) for the company class with the highest total withdrawal value?



In [1]:
import io
import numpy as np
import pandas as pd
import requests

# Fetch Filed IPO data
url = "https://www.iposcoop.com/ipos-recently-filed/"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML,"
        " like Gecko) Chrome/115.0.0.0 Safari/537.36"
    )
}
resp = requests.get(url, headers=headers)

df = pd.read_html(io.StringIO(resp.text))[0]
df_withdrawn = df[df["Expected To Trade"] == "Withdrawn"].copy()


# Company classification logic
def classify_company(name):
  if not isinstance(name, str):
    return "Other"
  if "Technologies" in name:
    return "Technologies"
  elif any(
      kw in name for kw in ["Acquisition Corp", "Acquisition Corporation", "Corp"]
  ):
    return "Acquisition Corp"
  elif any(kw in name for kw in ["Inc", "Incorporated"]):
    return "Inc."
  elif "Group" in name:
    return "Group"
  elif any(kw in name for kw in ["Ltd", "Limited"]):
    return "Limited"
  elif any(kw in name for kw in ["Holdings", "Holding"]):
    return "Holdings"
  else:
    return "Other"


df_withdrawn["Company Type"] = df_withdrawn["Company"].apply(classify_company)


# Helper functions to clean currency and string values
def parse_price(val):
  if pd.isna(val) or str(val).strip() in ["-", ""]:
    return np.nan
  clean_str = str(val).replace("$", "").replace(",", "").strip()
  try:
    return float(clean_str)
  except ValueError:
    return np.nan


def clean_num(val):
  if pd.isna(val) or str(val).strip() in ["-", ""]:
    return np.nan
  clean_str = str(val).replace("$", "").replace(",", "").strip()
  try:
    return float(clean_str)
  except ValueError:
    return np.nan


# Parse Prices and compute Avg_price
low_p = df_withdrawn["Price Low"].apply(parse_price)
high_p = df_withdrawn["Price High"].apply(parse_price)
df_withdrawn["Avg_price"] = (low_p + high_p) / 2.0
df_withdrawn["Avg_price"] = df_withdrawn["Avg_price"].fillna(low_p).fillna(high_p)

# Clean numeric volumes/shares
df_withdrawn["Shares (millions)"] = df_withdrawn["Shares (millions)"].apply(clean_num)
df_withdrawn["Est $ Vol (millions)"] = df_withdrawn["Est $ Vol (millions)"].apply(clean_num)

# Calculate Shares_offered_value
calc_val = df_withdrawn["Shares (millions)"] * df_withdrawn["Avg_price"]
df_withdrawn["Shares_offered_value"] = np.where(
    calc_val.notna(), calc_val, df_withdrawn["Est $ Vol (millions)"]
)

# Aggregation by Company Type
summary = (
    df_withdrawn.groupby("Company Type")["Shares_offered_value"]
    .sum()
    .reset_index()
)
summary = summary.sort_values(by="Shares_offered_value", ascending=False)

print("--- Withdrawn IPO Value by Company Type ---")
print(summary.to_string(index=False))

top_class = summary.iloc[0]["Company Type"]
top_val = summary.iloc[0]["Shares_offered_value"]
print(f"\nHighest Total Withdrawal Class: {top_class} with ${top_val:,.2f} million")

--- Withdrawn IPO Value by Company Type ---
    Company Type  Shares_offered_value
           Other             1115.4450
Acquisition Corp              499.9850
            Inc.              351.0000
        Holdings              311.6575
         Limited              219.2500
    Technologies              184.9000
           Group               56.5750

Highest Total Withdrawal Class: Other with $1,115.44 million


## Question 2: [IPO] Median Sharpe Ratio for 2025 IPOs (First 8 Months)


What is the median Sharpe ratio (as of 11 September 2026) for companies that went public before 1 September 2025?



In [2]:
import io
import numpy as np
import pandas as pd
import requests
import yfinance as yf

# Fetch 2025 IPO Pricings
url = "https://www.iposcoop.com/2025-pricings/"
headers = {"User-Agent": "Mozilla/5.0"}
resp = requests.get(url, headers=headers)
df_2025 = pd.read_html(io.StringIO(resp.text))[0]

# Clean 'Offer Date' and 'Return'
df_2025["Offer Date"] = pd.to_datetime(df_2025["Offer Date"], errors="coerce")
df_2025["Return_clean"] = (
    df_2025["Return"]
    .astype(str)
    .str.replace("%", "")
    .str.replace("$", "")
    .str.strip()
)
df_2025["Return_num"] = pd.to_numeric(df_2025["Return_clean"], errors="coerce")

# Filtering Offer Date < 2025-09-01 and Return != 0%
filtered_ipos = df_2025[
    (df_2025["Offer Date"] < "2025-09-01") & (df_2025["Return_num"] != 0)
].copy()
tickers = filtered_ipos["Symbol"].dropna().unique().tolist()
print(f"Filtered Active Tickers Count: {len(tickers)}")
data = yf.download(tickers, start="2025-01-01", end="2026-09-15", group_by="ticker")

# Feature engineering per stock
records = []
for ticker in tickers:
  try:
    if len(tickers) == 1:
      df_t = data.copy()
    else:
      if ticker in data.columns.levels[0]:
        df_t = data[ticker].dropna(how="all").copy()
      else:
        continue

    if df_t.empty or "Close" not in df_t.columns:
      continue

    # Flatten multiIndex columns if present
    if isinstance(df_t.columns, pd.MultiIndex):
      df_t.columns = df_t.columns.get_level_values(0)

    df_t = df_t.sort_index()
    df_t["Ticker"] = ticker

    # growth_252d = Close / Close.shift(252)
    df_t["growth_252d"] = df_t["Close"] / df_t["Close"].shift(252)

    # Annualized volatility = 30-day std * sqrt(252)
    df_t["volatility"] = df_t["Close"].rolling(30).std() * np.sqrt(252)

    # Sharpe ratio (Risk-free rate = 5.0%)
    df_t["Sharpe"] = (df_t["growth_252d"] - 0.05) / df_t["volatility"]

    records.append(df_t)
  except Exception as e:
    continue

stocks_df = pd.concat(records)

# Filter for target date '2026-09-11'
sept_11_data = stocks_df[stocks_df.index == "2026-09-11"]
stats = sept_11_data[["growth_252d", "volatility", "Sharpe"]].describe()

print("\n--- Summary Statistics as of September 11, 2026 ---")
print(stats)

median_sharpe = sept_11_data["Sharpe"].median()
print(f"\nMedian Sharpe Ratio: {median_sharpe:.4f}")

Filtered Active Tickers Count: 146


/tmp/ipykernel_555/874486676.py:30: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, start="2025-01-01", end="2026-09-15", group_by="ticker")
[*                      3%                       ]  5 of 146 completedERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TBH"}}}
[*********             18%                       ]  26 of 146 completedERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: EMPG"}}}
[*********************100%***********************]  146 of 146 completed
ERROR:yfinance:
15 Failed downloads:
ERROR:yfinance:['TBH', 'AHL', 'EMPG', 'MJID', 'SKBL', 'AGH', 'CAEP', 'SDM', 'CEPT', 'PTNM', 'EPWK', 'WGRX', 'MCTR', 'MTSR']: YFTzMissingError('possibly delisted; no timezone found')
ERROR:yfinance:['DLXY']: YFPricesMissingError('possibly delisted; no price data fou


--- Summary Statistics as of September 11, 2026 ---
Price  growth_252d  volatility      Sharpe
count   129.000000  130.000000  129.000000
mean      1.065215   27.725051         inf
std       3.059657   67.095064         NaN
min       0.001005    0.000000   -0.040147
25%       0.149038    2.325895    0.011942
50%       0.596026    7.927892    0.048009
75%       1.040233   23.659042    0.134248
max      33.638270  627.698106         inf

Median Sharpe Ratio: 0.0480


## Question 3: [IPO] 'Fixed Months Holding Strategy'


What is the optimal number of months (1 to 12) to hold a newly IPO'd stock in order to maximize the median growth value?



In [3]:
# Reusing stocks_df from Question 2
ticker_growth_list = []

for ticker, group in stocks_df.groupby("Ticker"):
  group = group.sort_index().copy()
  if group.empty:
    continue

  p0 = group["Close"].iloc[0]  # First trading day price
  growth_dict = {"Ticker": ticker}

  # Calculate 12 future growth horizons (1 month = 21 trading days)
  for m in range(1, 13):
    days = m * 21
    if len(group) > days:
      growth_dict[f"future_growth_{m}_m"] = group["Close"].iloc[days] / p0
    else:
      growth_dict[f"future_growth_{m}_m"] = np.nan

  ticker_growth_list.append(growth_dict)

growth_df = pd.DataFrame(ticker_growth_list)
medians = growth_df.filter(like="future_growth_").median()

print("--- Median Growth Ratio by Holding Period ---")
for col, val in medians.items():
  month_num = col.split("_")[2]
  print(f"Month {month_num:>2}: {val:.4f}")

best_month_col = medians.idxmax()
best_month = best_month_col.split("_")[2]
max_median_growth = medians.max()

print(f"\nOptimal Holding Period: {best_month} Month(s)")
print(f"Maximum Median Growth Value: {max_median_growth:.4f}")

--- Median Growth Ratio by Holding Period ---
Month  1: 0.9306
Month  2: 0.8917
Month  3: 0.8280
Month  4: 0.7360
Month  5: 0.6959
Month  6: 0.7334
Month  7: 0.6680
Month  8: 0.6040
Month  9: 0.5844
Month 10: 0.5335
Month 11: 0.4871
Month 12: 0.5083

Optimal Holding Period: 1 Month(s)
Maximum Median Growth Value: 0.9306


## Question 4: [Strategy] Simple RSI-Based Trading Strategy


What is the total profit (in $ thousands) you would have earned by investing $1000 every time a stock was oversold (RSI < 30)?



In [4]:
import gdown
import pandas as pd

# Download precomputed dataset
file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
gdown.download(
    f"https://drive.google.com/uc?id={file_id}", "data.parquet", quiet=False
)

df = pd.read_parquet("data.parquet", engine="pyarrow")

# Identify date and RSI columns
date_col = "Date" if "Date" in df.columns else "date"
rsi_col = "RSI" if "RSI" in df.columns else "rsi"

df[date_col] = pd.to_datetime(df[date_col])

# Filter trades: 2000-01-01 to 2025-06-01 where RSI < 30
trades = df[
    (df[date_col] >= "2000-01-01")
    & (df[date_col] <= "2025-06-01")
    & (df[rsi_col] < 30)
].copy()

# Calculate profit per trade
# Net Income = $1000 * (growth_future_30d - 1)
trades["net_income"] = 1000 * (trades["growth_future_30d"] - 1)
total_net_income = trades["net_income"].sum()
total_net_income_thousands = total_net_income / 1000.0

print(f"Total Oversold Trades (RSI < 30): {len(trades):,}")
print(f"Total Net Income: ${total_net_income:,.2f}")
print(
    f"Net Income ($ Thousands): ${total_net_income_thousands:,.2f} thousand"
)

Downloading...
From (original): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-
From (redirected): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-&confirm=t&uuid=6fa4b983-ef3c-46cf-a3fa-eaeca2aa29b3
To: /content/data.parquet
100%|██████████| 130M/130M [00:07<00:00, 17.4MB/s]


Total Oversold Trades (RSI < 30): 5,206
Total Net Income: $65,805.59
Net Income ($ Thousands): $65.81 thousand
